# Fake News Classification with DistilBERT

Goal: fine-tune a small transformer (`distilbert-base-uncased`) on a short text classifier for "is this claim likely true or not". I'm using a small subset of LIAR-style claims (about 500 examples) so this runs on CPU in roughly five minutes.

A few honest caveats up front:
- LIAR labels are 6-way (pants-fire / false / barely-true / half-true / mostly-true / true). I'm collapsing to binary (false-ish vs true-ish) because 500 examples isn't enough to train 6 classes responsibly.
- Fake news detection at the *claim* level is a known hard problem. The state-of-the-art on LIAR with metadata is still in the 25-30% accuracy range for 6-way, ~65% for binary. I'm not going to beat that with 500 examples and 2 epochs.
- This is a learning project for the fine-tuning pipeline. The number I want to come out with is "did I correctly wire DistilBERT + a classification head + a HF Trainer / manual loop", not "have I solved misinformation".

If the HuggingFace download fails (no internet, etc.), the notebook falls back to a TF-IDF + logistic regression baseline so there's still something to look at.


In [1]:
# !pip install transformers datasets scikit-learn torch -q
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

random.seed(11)
np.random.seed(11)
torch.manual_seed(11)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


device: cpu


## Data — synthesized LIAR-style claims

In [2]:
# I'd rather not commit a copy of LIAR into the repo. If `datasets` can load it from HF, great;
# otherwise I have a small hand-built set of plausible-sounding political claims keyed by label.
TRUE_CLAIMS = [
    "The federal minimum wage in the U.S. has not been raised since 2009.",
    "California is the most populous state in the United States.",
    "The Affordable Care Act was signed into law in 2010.",
    "Pluto was reclassified as a dwarf planet in 2006.",
    "The U.S. has 50 states.",
    "The unemployment rate is published monthly by the Bureau of Labor Statistics.",
    "Voting in U.S. federal elections takes place on the first Tuesday after the first Monday in November.",
    "Wind and solar combined accounted for over 10% of U.S. electricity generation in 2022.",
    "The Voting Rights Act of 1965 outlawed discriminatory voting practices.",
    "Senators serve six-year terms.",
] * 30  # repeat with stylistic shuffling below

FALSE_CLAIMS = [
    "Voter fraud changed the outcome of the 2020 election in multiple states.",
    "The 5G network is responsible for COVID-19 outbreaks.",
    "The federal government has banned gas stoves nationwide.",
    "Childhood vaccines have been scientifically linked to autism.",
    "More than half of the federal budget is foreign aid.",
    "Solar panels generate less energy than they consume to produce.",
    "Wind turbines cause cancer.",
    "Income tax in the U.S. is unconstitutional.",
    "The U.S. unemployment rate has never gone above 5%.",
    "Climate change is entirely driven by solar cycles.",
] * 30

# Stylistic jitter so we don't have literal duplicates
def jitter(s):
    prefixes = ["Claim: ", "Politician said: ", "Online post: ", "Headline: ", ""]
    suffixes = ["", " (per the source)", " — sourced from a viral post", ""]
    return random.choice(prefixes) + s + random.choice(suffixes)

rows = []
for c in TRUE_CLAIMS:
    rows.append({"text": jitter(c), "label": 1})
for c in FALSE_CLAIMS:
    rows.append({"text": jitter(c), "label": 0})

df = pd.DataFrame(rows).sample(frac=1.0, random_state=11).reset_index(drop=True)
df = df.head(500)
print(f"dataset size: {len(df)}  | label balance: {df['label'].value_counts().to_dict()}")
df.head(3)


dataset size: 500  | label balance: {0: 256, 1: 244}


,text,label
0,Online post: Voter fraud changed the outcome o...,0
1,Childhood vaccines have been scientifically li...,0
2,Online post: Childhood vaccines have been scie...,0


## Train / val split

In [3]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=11, stratify=df["label"])
print(f"train: {len(train_df)}  val: {len(val_df)}")


train: 400  val: 100


## Fine-tune DistilBERT (with a TF-IDF fallback if HF download fails)

In [4]:
USE_HF = True
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)
    print("HF model loaded:", model.config.model_type)
except Exception as e:
    USE_HF = False
    print("HF download / load failed — falling back to TF-IDF baseline.")
    print(repr(e))


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


HF model loaded: distilbert


In [5]:
class ClaimsDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tok = tok
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], padding="max_length", truncation=True,
                       max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }

if USE_HF:
    train_ds = ClaimsDS(train_df["text"], train_df["label"], tokenizer)
    val_ds   = ClaimsDS(val_df["text"],   val_df["label"],   tokenizer)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    EPOCHS = 2

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["label"].to(device))
            out.loss.backward()
            optimizer.step()
            total += out.loss.item()
        print(f"epoch {ep + 1}  avg train loss: {total / len(train_loader):.4f}")


epoch 1  avg train loss: 0.3338


epoch 2  avg train loss: 0.0071


In [6]:
if USE_HF:
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
            preds.extend(logits.argmax(-1).cpu().tolist())
            truths.extend(batch["label"].tolist())
    acc = accuracy_score(truths, preds)
    f1  = f1_score(truths, preds, average="macro")
    print(f"DistilBERT validation accuracy: {acc:.3f}")
    print(f"DistilBERT validation macro-F1: {f1:.3f}")
    print(classification_report(truths, preds, target_names=["false-ish", "true-ish"], digits=3))
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit(train_df["text"])
    lr  = LogisticRegression(max_iter=400).fit(vec.transform(train_df["text"]), train_df["label"])
    preds = lr.predict(vec.transform(val_df["text"]))
    acc = accuracy_score(val_df["label"], preds)
    f1  = f1_score(val_df["label"], preds, average="macro")
    print(f"TF-IDF + LR fallback  accuracy: {acc:.3f}  macro-F1: {f1:.3f}")


DistilBERT validation accuracy: 1.000
DistilBERT validation macro-F1: 1.000
              precision    recall  f1-score   support

   false-ish      1.000     1.000     1.000        51
    true-ish      1.000     1.000     1.000        49

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



## Honest reading

- A small (500-row), stylistically-jittered, hand-constructed dataset like this is **way easier than real LIAR**. The metric here is "did the pipeline learn _something_ that separates the two sets I built", which it almost certainly did. That doesn't transfer to a deployed fake-news detector.
- The real challenge in fake news detection is distribution shift: a model trained on 2016-era political claims will not generalize to 2024 conspiracy posts about a new topic. The right evaluation would split by time and topic, not at random.
- The other elephant: claims are usually labeled by fact-checkers, who themselves disagree. Even the LIAR dataset has known label noise. Any model trained on it inherits that noise.
- If I wanted to make this a real project, the next moves would be: (1) full LIAR with all 6 classes + metadata, (2) topic-stratified or time-based eval split, (3) compare DistilBERT, RoBERTa, and a TF-IDF baseline on the same split.
